# COT positioning in Python

CFTC Commitments of Traders: list markets, pull history and compute net speculative positioning and its percentile.

Guide: https://finzdata.com/guides/cot-data-python

Set `FINZDATA_API_KEY` first (free key: https://api.finzdata.com/). Cells marked as needing a look-back bundle return HTTP 403 on free keys.

In [ ]:
%pip install -q finzdata

## Listing Available COT Markets

In [ ]:
import finzdata as yf
c = yf.Client()
markets = c.cot_markets(report='legacy_fo')
gold_markets = markets[markets['market'].str.contains('GOLD', case=False)]
print(gold_markets[['code', 'market']].head())

## Fetching Historical COT Data

In [ ]:
import finzdata as yf
c = yf.Client()
df = c.cot(report='legacy_fo', code='088691', start='2020-01-01')
print(df[['date', 'noncomm_positions_long_all', 'noncomm_positions_short_all']].head())

## Calculating Net Speculative Positioning

In [ ]:
import finzdata as yf
c = yf.Client()
df = c.cot(report='legacy_fo', code='088691', start='2020-01-01')
df['net_speculative'] = df['noncomm_positions_long_all'] - df['noncomm_positions_short_all']
print(df[['date', 'net_speculative']].tail())

## Computing Percentile Rank Over Lookback

In [ ]:
import finzdata as yf
import pandas as pd
c = yf.Client()
df = c.cot(report='legacy_fo', code='088691', start='2020-01-01')
df['net_speculative'] = df['noncomm_positions_long_all'] - df['noncomm_positions_short_all']
df['speculative_pct_rank'] = df['net_speculative'].rolling(260).rank(pct=True) * 100
print(df[['date', 'net_speculative', 'speculative_pct_rank']].tail(10))

## Handling Different COT Report Types

In [ ]:
import finzdata as yf
c = yf.Client()
# Example: TFF report for S&P 500 Consolidated
df_tff = c.cot(report='tff_fo', market='S&P 500 Consolidated', start='2025-01-01')
df_tff['net_asset_mgr'] = df_tff['asset_mgr_positions_long'] - df_tff['asset_mgr_positions_short']
print(df_tff[['date', 'net_asset_mgr']].head())